# report10 — 검출기 교정 ① — **오경보율(false alarm) 눈금 맞추기**

**핵심.** 드론 탐지에 쓰는 **CA-CFAR 검출기의 오경보율 눈금을 교정한다** — 명목 오경보율 대비 실제 발화율을 신호(WiFi·LTE·5G)별로 재고, 공정 비교에 필요한 신호별 교정표를 세운다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna 는 파형(PHY)·전파(RT)까지만 준다 — **표적 유무를 판정하는 검출기(레이더/센싱 모듈)가 없다.** 오경보율을 고정하는 CFAR 도, 그 전단의 직접파 제거(ECA)·교차모호함수(CAF)도 밖에서 가져와야 한다(§1). |
| **② 선행 연구의 방식** | 하드웨어 패시브 레이더 선행이 공통으로 쓰는 표준 검출 사슬은 **ECA→CAF→CA-CFAR** 이다 — 5G NR OFDM 패시브 레이더(Sensors 2026, DOI 10.3390/s26041317), 5G SSB 저고도 표적 패시브 검출(SPAWC 2025, arXiv:2504.02641), LTE450 패시브 레이더(IET RSN 2025, DOI 10.1049/rsn2.70092). CA-CFAR 문턱 공식은 교과서 표준(Richards)이다(§3). |
| **③ 쓴 라이브러리·결합** | 검출단을 새로 짜지 않고 오픈소스 **pyAPRiL**(GPLv3)의 ECA/CAF/**`caCfar.CA_CFAR`** 를 그대로 쓴다(중복 구현 회피). Sionna 가 만든 기준·감시 신호를 그대로 넘겨 WiFi·LTE·5G 세 모드 표적을 정답 거리빈에 검출함을 확인했다(`outputs/verify_pyapril.json`). 대규모 오경보율 측정만 규모를 GPU 몬테카를로로 키우되 표준 α 식과 소수점 15자리까지 일치시킨다(§4). |
| **④ 검증** | 명목 Pfa 대비 **경험 Pfa**. 이상적 백색 잡음 500,000장에서 배율 **0.997**(오차 1~2%)·α 공식 상대오차 8e-16 → 검출기 산수는 무죄(§2.1). 실제 지도에선 WiFi 1.45·LTE 2.66·5G 1.52배로 어긋나고, 이웃 칸 상관을 끄면 1.0 으로 복구된다(§2.4). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 명목 vs 경험 오경보율 (파형별) | outputs/verify_cfar.json — 파형별 8,000장 잡음 지도 | **우리 측정** |
| 이상적 백색 지도 검증 | outputs/verify_cfar.json — 이상적 잡음 500,000장 | **우리 측정** |
| CA-CFAR 문턱 공식 α(N, Pfa) | Richards, *Fundamentals of Radar Signal Processing* — 셀평균 CFAR 표준 | 문헌 |
| 셀 상관(whiteness) ρ_range · ρ_doppler | outputs/verify_cfar.json — 지도의 이웃칸 상관계수 | **우리 측정** |
| 슬로타임 Hann 창 · 정합필터 | src/passive_process.py — 거리-도플러 지도 형성 | 설계값 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: GPU 1장(자동선택). 파형당 8,000장 + 이상적 지도 50만 장의 몬테카를로. 수 분.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 오경보율 눈금 검증 — 파형별로 잡음 지도를 대량 생성해 CFAR 발화를 세고 JSON 을 남긴다
cd benchmark && ~/.venvs/py312/bin/python verify_cfar.py   # → outputs/verify_cfar.json

# 6패널 교정 그림 + 이 노트북
~/.venvs/py312/bin/python src/viz_report4.py       # → outputs/figures/report4_e1_cfar.png
~/.venvs/py312/bin/python src/make_notebook10.py   # → report10.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/verify_cfar.json` | 파형별 명목↔경험 오경보율 · 이상적 백색 검증 · 셀 상관 · 대조실험 |
| `outputs/figures/report4_e1_cfar.png` | CFAR 오경보율 교정 6패널(재사용) |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **이 리포트는 '어느 신호가 좋다'를 말하지 않는다.** 오직 **검출기의 눈금**만 다룬다. 신호 간 탐지 성능의 공정 비교는 이 교정을 전제로 한 뒤에야 가능하다.
- **측정 오경보율에는 통계적 불확실성이 있다.** 명목 1e-06 처럼 깊은 곳에서는 지도당 발화가 몇 개뿐이라 배율의 신뢰구간이 넓다. 그림 (b) 의 오차막대가 그것이다.
- **셀 상관 → 오경보 증가는 이 처리사슬에 고유한 성질이다.** 다른 창(window)·다른 과표본율을 쓰면 배율이 달라진다. 그래서 교정은 **설정마다** 다시 재야 한다 (§4).
- **여기서 다루는 눈금은 잡음-only 지도 위의 것이다.** 실제 클러터·유령이 얹히면 오경보의 원천이 더 늘 수 있다(→ report09 의 유령). 그 위의 검출 눈금은 별개 문제다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report09 (앞) | 바닥 유령이 '별개 표적'으로 물리적으로 분해되는 것을 봤다 — 그 유령을 검출기가 실제로 몇 번 발화하는지는 검출기 눈금에 달려 있다 |
| report11 (다음) | 검출기 교정의 **나머지 눈금들** — 저속 표적 맹점·거리 분해능·위치 관측성 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **오경보(false alarm)** | 표적이 없는데도 검출기가 '있다'고 발화하는 것. 잡음 봉우리를 표적으로 착각 |
| **오경보율 Pfa** | 빈 칸 하나가 발화할 확률. 'Pfa=$10^{-4}$' = 잡음 칸 만 개 중 하나꼴 오발화 |
| **명목(nominal) Pfa** | 우리가 검출기에 **요구한** 오경보율 |
| **경험(empirical) Pfa** | 실제로 재보니 **나온** 오경보율. 발화 수 ÷ 전체 빈 칸 수 |
| **배율(ratio)** | 경험 Pfa ÷ 명목 Pfa. 1.0 이면 눈금이 정확, 2.66 이면 요구보다 2.66배 자주 발화 |
| **5G NR 100MHz (이 리포트)** | 전대역(98 MHz) 기준신호 — NR-PRS/풀점유 — 를 뜻한다. 늘 켜진 SSB(7.2 MHz)만 쓰는 5G 상시 모드는 대역이 훨씬 좁아 이 교정이 그대로 적용되지 않는다(report12 의 G1 모드가 그 경우) |
| **CFAR** | Constant False Alarm Rate. 표적 후보 칸 주변의 잡음 수준을 보고 문턱을 **스스로** 정하는 검출기. 잡음이 세지면 문턱도 같이 올려 오경보율을 일정하게 유지하려는 장치 |
| **가드/트레이닝 셀** | 후보 칸 둘레의 이웃 칸. 바깥쪽 **트레이닝** 칸으로 잡음을 추정하고, 사이의 **가드** 칸은 표적 에너지가 새어들지 않게 비워 둔다 |
| **거리-도플러(RD) 지도** | 가로축=거리, 세로축=속도(도플러)인 2차원 지도. 각 칸이 그 거리·속도에 반사가 있나를 나타낸다. 검출은 이 지도 위에서 봉우리를 찾는 일 |
| **Hann 창** | 신호를 도플러로 변환하기 전 양 끝을 부드럽게 눌러 옆 속도로의 번짐을 줄이는 가중창. 부작용으로 **이웃 도플러 칸끼리 닮게** 만든다 |
| **상관계수 ρ** | 두 칸이 얼마나 함께 움직이나. 0=무관(독립), 1=완전히 같음. CFAR 은 트레이닝 칸이 서로 **독립**이라 가정하는데, ρ 가 크면 그 가정이 깨진다 |
| **유효 독립 표본** | 겉보기 칸 수 중 **실제로 독립인** 몫. 이웃이 닮으면(ρ↑) 24칸을 봐도 실효는 절반뿐 — 잡음 추정이 실제보다 촘촘해 보여 문턱이 낮아진다 |
| **과표본(oversampling)** | 신호 대역폭이 좁은데 표본을 촘촘히 뜨면, 이웃 거리 칸이 서로 닮는다. 좁은 대역(LTE)일수록 심하다 |

</details>

---


# §1. Sionna 의 공백 — 판정할 검출기가 없다

Sionna 는 파형(PHY)과 전파(RT)까지는 준다 — 그 부분은 report05 에서 검증했다(채널 NMSE −135 dB). 하지만 Sionna 에는 **표적을 판정하는 검출기가 없다**(공식 레이더/센싱 모듈 부재). 레이더 검출은 결국 **거리-도플러(RD) 지도** 위에서 봉우리를 찾는 일이다 — 가로축은 거리, 세로축은 속도(도플러), 각 칸이 '여기에 반사가 있나?'를 나타낸다. 이 판정 장치는 Sionna 밖에서 가져와야 한다.

검출기는 각 칸의 값이 **문턱**을 넘으면 '표적!'이라고 판정한다. 문턱을 낮게 잡으면 약한 표적도 잡지만 빈 칸의 잡음 봉우리에도 자주 속는다 — 이 헛발화가 **오경보(false alarm)** 다. 문턱을 높이면 오경보는 줄지만 진짜 표적도 놓친다. 그래서 검출은 **오경보율(Pfa)** 을 먼저 고정한다: 빈 칸이 잘못 발화할 확률을 만분의 1($10^{-4}$)로 유지하라.

### CFAR — 문턱을 스스로 정하는 검출기

잡음의 세기는 상황마다 다르다. 문턱을 고정된 숫자로 박아두면 잡음이 조금만 세져도 오경보가 폭발한다. **CFAR**(Constant False Alarm Rate)은 이걸 피한다: 후보 칸 **둘레의 이웃 칸**을 보고 '지금 이 근방의 잡음이 이 정도구나'를 추정한 뒤, 그 추정값에 정해진 배수 α 를 곱해 문턱을 만든다. 잡음이 세지면 이웃도 같이 세지므로 문턱도 따라 올라 — 오경보율이 **일정하게** 유지된다(는 것이 약속).

- **트레이닝 셀**: 후보 칸 바깥 둘레의 이웃 칸들. 여기서 잡음 세기를 추정한다.
- **가드 셀**: 후보와 트레이닝 사이의 빈 띠. 표적 에너지가 트레이닝으로 새어들어 잡음 추정을 오염시키지 못하게 막는다.

표준 설정은 가드 2칸·트레이닝 6칸(`g2x2_t6x6`)이다. 셀평균 CFAR 의 문턱 배수 α 는 트레이닝 칸 수 N 과 목표 Pfa 로 정해지는 **교과서 닫힌형 공식**(α = N·(Pfa^(−1/N) − 1), Richards)을 그대로 쓴다.

**핵심 가정.** CFAR 의 α 공식은 트레이닝 칸들이 서로 **독립**이라고 가정한다. 이 가정이 이 리포트의 주인공이다 — 이게 깨지면(이웃이 서로 닮으면) 눈금이 어긋난다. 남는 질문은 하나다: 이 표준 문턱이 실제로 약속한 오경보율을 내는가? 그것을 §2 가 측정으로 답한다.

---
# §2. 측정 — 눈금이 언제 맞고 언제 어긋나나

'표준 문턱이 약속한 오경보율을 내는가'를 말이 아니라 측정으로 답한다. 먼저 이상적 잡음에서 검출기 자체가 멀쩡함을 못박고(§2.1), 실제 처리사슬이 만든 지도에서 어긋남을 드러내고(§2.2), 그 원인을 이웃 칸 상관으로 지목한 뒤(§2.3), 상관을 직접 꺼서 인과를 확정한다(§2.4).

## 2.1 이상적 잡음에서는 눈금이 정확하다 — 검출기 산수는 무죄

판정: **확정** — 이상적 백색 잡음 500,000장에서 경험 Pfa ÷ 명목 Pfa = **0.997** (오차 1~2%).

**① 표준 문턱 공식을 그대로 쓴다.** 위 교과서 닫힌형 α(Richards)를 표준 공식과 대조하면 상대오차 **8e-16** — 소수점 15자리까지 같다. 문턱 계산은 표준을 **정확히** 따른다.

**② 이상적 잡음 위에서 약속을 지킨다.** 서로 아무 상관 없는(완전 백색) 복소 가우시안 잡음으로만 채운 지도 500,000장(총 564,000,000칸 규모)에 이 표준 CFAR 를 물려 발화 수를 센다. 명목 대비 실제 발화 배율:

| 명목 Pfa | 경험 Pfa | 배율 |
|---|---|---|
| $10^{-6}$ | 9.96e-07 | **0.996배** |
| $10^{-5}$ | 1.02e-05 | **1.018배** |
| $10^{-4}$ | 9.97e-05 | **0.997배** |
| $10^{-3}$ | 1.00e-03 | **1.000배** |
| $10^{-2}$ | 9.99e-03 | **0.999배** |

> 명목이 만분의 1이든 백만분의 1이든, 실제 발화는 요구값의 **1.00배 안쪽**에서 논다. 이상적 조건에서 검출기는 약속을 지킨다. **그래서 이제부터 나올 어긋남은 '검출기 버그'가 아니다** — 산수는 무죄다. 문제는 입력으로 들어오는 지도가 이상적 잡음이 아니라는 데 있다.

---
## 2.2 실제 지도에서는 신호마다 다르게 어긋난다

이상적 잡음이 아니라 **실제 처리사슬이 만든 거리-도플러 지도** 위에서 같은 검출기를 파형별로 8,000장씩 다시 잰다.

판정: **확정** — 명목 $10^{-4}$ 에서 실제 발화는 **WiFi 1.45 · LTE 2.66 · 5G 1.52배**. 어긋남이 **파형마다 다르다.**

![E1 CFAR false-alarm calibration (6 panels)](outputs/figures/report4_e1_cfar.png)

> 위 6패널은 이 리포트 전체의 그림이다. **(a)** 명목↔경험 곡선(회색=이상적 백색 지도, 색선=실제 사슬), **(b)** 파형별 배율과 신뢰구간, **(c)~(d)** 원인 두 가지, **(e)** 도플러 마스크 민감도, **(f)** 진짜 오경보율로 다시 그린 검출곡선. 아래에서 하나씩 읽는다.

![CFAR threshold sweep on a real 5G range-Doppler map](outputs/renders/anim/cfar_sweep_nr.gif)

> **움직이는 그림:** 실제 5G(NR100) 거리-도플러 지도 위에서 CFAR 문턱을 낮은 쪽부터 쓸어 올린다. 문턱이 내려갈수록 표적 봉우리가 먼저 살아나고, 더 내리면 잡음 바닥까지 발화(오경보)가 번진다 — **명목 오경보율 한 점이 실제로는 어느 문턱에 대응하는지**를 눈으로 보여준다.

### 실제 발화는 요구보다 많다 — 그리고 그 정도가 신호마다 다르다

그림 (b) 의 막대다. 명목 $10^{-4}$, 운영 설정에서 실제로 잰 배율(막대=점추정, 수염=95% 신뢰구간):

| 파형 | 대역폭 | 경험 Pfa | 배율 (경험÷명목) |
|---|---|---|---|
| **WiFi 80MHz** | 77 MHz | 1.45e-04 | **1.45배** |
| **LTE 20MHz** | 18 MHz | 2.66e-04 | **2.66배** |
| **5G NR 100MHz** | 98 MHz | 1.52e-04 | **1.52배** |

> 세 신호 모두 요구보다 자주 발화하지만, **LTE 는 5G 보다 1.8배 느슨한 문턱**을 쓰고 있다. 같은 '만분의 1'을 주문했는데 LTE 는 사실상 그보다 훨씬 헐거운 기준으로 표적을 외치는 것이다.

### 그래서 무엇이 문제인가 — 애초에 **불공정한 경기**였다

"어느 신호가 드론을 더 잘 잡나"라는 벤치마크의 표준 방식은, **오경보율을 같은 값으로 고정**해 놓고 탐지율(Pd)을 비교하는 것이다. 그래야 '같은 실수 예산에서 누가 더 잘 잡나'라는 공정한 질문이 된다.

그런데 방금 본 대로, **명목** 오경보율을 같게 맞춰도 **진짜** 오경보율은 신호마다 다르다. 명목 $10^{-4}$ 에 세 신호를 세워도, 실제로는 LTE 가 5G 보다 1.8배 헐거운 문턱에서 달리고 있다. **느슨한 문턱은 탐지율을 공짜로 올려준다** — 오경보를 더 허용하는 대신 표적도 더 잡으니까. 즉 명목 Pfa 로 줄을 세운 비교는 LTE 에게 몰래 유리한 핸디캡을 준 셈이다.

**명목 오경보율에서의 탐지율 비교는, 실제로는 서로 다른 오경보율에서의 비교였다.** 벤치마크의 핵심 질문이 처음부터 공정하게 물어지지 않았던 것이다. 그래서 겨루기 전에 **눈금부터** 맞춰야 한다.

> 그림 (f) 는 이 점을 정직하게 보여준다: 가로축을 명목이 아니라 **실제로 측정된 오경보율**로 바꿔 검출곡선을 다시 그린 것이다. 진짜 오경보율 위에서 비교해야 공정하다.

---
## 2.3 왜 어긋나나 — 이웃 칸이 서로 닮아서

판정: **확정** — CFAR 의 '트레이닝 칸은 독립' 가정이 깨진다. 도플러축은 **모든 신호 공통**으로, 거리축은 **대역이 좁을수록** 이웃 칸이 닮는다.

CFAR 의 문턱 공식은 트레이닝 칸 N 개가 서로 **독립인 잡음 표본**이라고 가정한다. 그래야 N 개를 평균한 잡음 추정이 충분히 안정적이고, 그 안정성에 맞춰 문턱 배수 α 가 계산된다. 그런데 이웃 칸이 서로 **닮아 있으면(상관 ρ > 0)**, 겉보기엔 N 개를 봐도 실제 독립 정보는 그보다 적다 — **유효 독립 표본**이 줄어든다는 뜻이다.

> **직관 — 답을 베낀 여론조사.** 24명에게 물어 여론을 재는데 그중 절반이 옆 사람 답을 베꼈다면, 표본은 24개지만 실제 정보는 12개뿐이다. 그런데 조사자는 여전히 '24명이나 물었으니 추정이 촘촘하다'고 **과신**한다. CFAR 도 똑같이 과신해서 문턱을 **너무 낮게** 잡고, 결국 헛것을 더 자주 본다.

지도의 이웃 칸 상관 ρ 를 두 축에서 각각 쟀다(바로 옆 칸과의 상관계수, 0=독립·1=완전 동일):

| 파형 | 도플러축 ρ (세로) | 거리축 ρ (가로) | 유효 독립 표본 몫(2D) |
|---|---|---|---|
| **WiFi 80MHz** | +0.46 | +0.00 | 0.49 |
| **LTE 20MHz** | +0.46 | +0.28 | 0.31 |
| **5G NR 100MHz** | +0.46 | +0.07 | 0.40 |

두 개의 서로 다른 이야기가 이 표에 있다 — 도플러축의 공통 손해와, 거리축의 신호별 손해.

### 도플러축: 모든 신호에 공통 — Hann 창의 부작용

도플러축 상관은 세 신호 모두 거의 같은 **+0.46** 이다. 신호 종류와 무관하다는 건 **원인이 신호가 아니라 처리 과정에 있다**는 신호다. 범인은 **슬로타임 Hann 창**이다. 여러 펄스를 모아 속도(도플러)를 뽑을 때 그 앞에 Hann 창을 곱하는데 — 도플러 봉우리가 옆 속도로 번지는 곁잎(사이드로브)을 줄이려는 표준 기법이다 — 그 대가로 창은 **이웃 도플러 칸끼리 서로 닮게** 만든다. 그래서 도플러축 유효 독립 표본이 약 절반으로 준다(세 신호 모두 ρ≈+0.46).

> 이건 **모든 신호가 똑같이 지불하는 공통 비용**이다. 그래서 세 신호를 다 위로(배율>1) 밀어 올린다. 다만 이것만으로는 신호 간 **차이**를 설명하지 못한다. 그 차이는 거리축에서 온다.

### 거리축: 대역이 좁을수록 심하다 — 과표본 상관

거리축 상관은 신호마다 극적으로 다르다:

- **WiFi(80MHz)**: ρ ≈ +0.00 — 거의 완전 독립.
- **5G(100MHz)**: ρ ≈ +0.07 — 약간 닮음.
- **LTE(20MHz)**: ρ ≈ +0.28 — 이웃 거리 칸이 뚜렷이 닮음.

이유는 **과표본**이다. 거리 분해능은 대역폭이 정한다 — 대역이 좁으면 실제로 구분되는 거리 간격이 넓다. 그런데 우리는 표본을 촘촘히 뜨므로, 좁은 대역(LTE)에서는 이웃 거리 칸들이 사실상 **같은 넓은 덩어리를 겹쳐 보는** 꼴이 된다 → 서로 닮는다. 넓은 대역(WiFi)은 칸마다 다른 거리를 또렷이 보므로 이웃이 안 닮는다.

**왜 LTE 가 제일 나쁜가:** 도플러축 손해는 셋 다 같지만, LTE 는 **거기에 거리축 손해까지 겹친다.** 두 축의 상관이 곱해져 2D 유효 독립 표본이 WiFi 0.49 · 5G 0.40 인데 LTE 는 0.31 까지 떨어진다 — 그래서 배율도 LTE 가 2.66 로 가장 크다. 그림 (c)(d) 가 이 두 원인을 각각 떼어 보여준다.

---
## 2.4 증명 — 상관을 끄면 눈금이 돌아온다

"이웃 상관이 원인"은 아직 **상관관계**일 뿐이다. **인과**를 못박으려면 상관을 직접 꺼 보고 눈금이 돌아오는지 봐야 한다. 5G 지도에서 두 원인을 하나씩 제거한다.

판정: **확정** — 두 상관을 모두 끄면 배율이 명목값(1.0)으로 복구된다.

원인으로 지목한 두 상관을 각각 없애는 표준 DSP 변형 두 가지로 지도를 다시 형성한다.

- **도플러 상관 끄기**: 슬로타임 Hann 창을 **직사각 창(rect)**으로 바꾼다 → 도플러축 상관이 사라진다.
- **거리 상관 끄기**: 정합필터 대신 **백색화(부정합) 필터**를 쓴다 → 거리축 색칠이 사라진다.

| 설정 | 배율 (명목 $10^{-4}$) | 도플러 유효독립 |
|---|---|---|
| 기준 (Hann + 정합필터) | **1.25배** | 0.50 |
| Hann 제거 (rect) | **0.96배** ← 복구 | 0.99 |
| 거리 백색화 | 1.28배 (홀로는 거의 그대로) | — |
| **둘 다 제거** | **1.02배** ← 완전 복구 | — |

> **Hann 을 직사각창으로 바꾸자** 도플러축 상관이 +0.46 → ≈0.0004 로 사라지고, 유효 독립 표본이 0.50 → 0.99 로 회복되며, 배율이 1.25 → 0.96 로 **명목값을 되찾는다.** 거리 백색화까지 겹치면 1.02 로 완전히 복구된다.

**이것이 인과를 못박는다.** 상관을 켜면 눈금이 어긋나고, 끄면 돌아온다. 어긋남의 원인은 **정확히 이웃 칸 상관**이다 — 도플러축(Hann, 공통)과 거리축(과표본, 대역이 좁을수록)의 곱. 검출기 산수(§2.1)도, 신호의 물리도 아니고, **지도가 매끈해 이웃이 닮았다**는 것이 전부다.

---
# §3. 선행 연구의 방식 — 패시브 레이더 표준 검출 사슬

이 검출기는 우리가 발명한 것이 아니다. 표적 유무 판정은 **패시브 레이더의 표준 처리사슬**로 밖에서 가져온다: 직접파를 지우는 **ECA**, 지연·도플러를 재는 **CAF(교차모호함수)**, 그 지도 위에서 문턱으로 판정하는 **CFAR**. 실제 하드웨어 선행(USRP 로 셀룰러·WiFi 조명원 표적을 잡은 패시브 레이더)이 쓰는 것도 정확히 이 ECA→CAF→CFAR 구성이다:

| 선행 | 조명원 | 검출 사슬 |
|---|---|---|
| 5G NR OFDM 패시브 레이더 (Sensors 2026, DOI 10.3390/s26041317) | 5G NR 하향 | ECA→CAF→CFAR |
| 5G SSB 저고도 표적 패시브 검출·측위 (SPAWC 2025, arXiv:2504.02641) | 5G SSB | 상관→CFAR |
| LTE450 기반 패시브 레이더 드론 탐지 (IET RSN 2025, DOI 10.1049/rsn2.70092) | LTE450 | ECA→CAF→CFAR |

CA-CFAR 의 문턱 공식(α = N·(Pfa^(−1/N)−1))은 특정 논문의 것이 아니라 교과서 표준이다(Richards, *Fundamentals of Radar Signal Processing*). 즉 우리가 쓰는 검출기는 **표준**이고, 선행이 실측으로 동작을 확인한 바로 그 구성이다. 이 리포트는 그 표준 검출기의 마지막 관문 — CFAR 의 **오경보율 눈금** — 만 잰다(검출기 교정의 나머지 — 저속 표적·거리 분해능·위치 관측성 — 은 report11 소관).

---
# §4. 우리가 쓴 방식 — pyAPRiL 검출단과 신호별 교정표

### 검출단은 오픈소스를 그대로 쓴다 (중복 구현 회피)

검출 사슬을 새로 짜지 않는다. 오픈소스 **pyAPRiL**(GPLv3)이 ECA/ECA-S·CAF·**`caCfar.CA_CFAR`**·DoA 를 그대로 제공한다 — 실제 하드웨어 패시브 레이더 연구가 쓰는 바로 그 구성이다. Sionna 가 만든 기준·감시 신호를 pyAPRiL 에 그대로 넘겨 돌리면 WiFi·LTE·5G 세 모드 모두 표적을 정답 거리빈에 검출한다(`outputs/verify_pyapril.json`). 즉 우리가 쓰는 검출기는 표준이고, 표준 구현으로 동작이 확인된 것이다.

다만 이 리포트의 **대규모 오경보율 측정**(파형당 8,000장 + 이상적 지도 수십만 장)은 규모를 **GPU 몬테카를로**로 키운다. 표준 CA-CFAR α 식과 소수점 15자리까지 같은 검출(§2.1, 상대오차 8e-16)을 규모만 GPU 로 키운 것이며, pyAPRiL 이 이 배율 곡선 자체를 계산한 것은 아니다. 검증의 기준은 라이브러리 대조가 아니라 **명목 Pfa 대비 경험 Pfa** 다 — 이상적 잡음에서 배율 0.997(§2.1), 실제 지도에서 신호별 어긋남과 상관 제거 시 복구(§2.2~§2.4).

### 처방 — 신호별 교정표

이웃 상관은 처리사슬의 구조에서 나온다(Hann 창은 곁잎을 줄이려고 일부러 쓰는 것이고, 과표본도 설계의 일부다). 그래서 어긋남 자체를 없애기보다, **각 신호가 얼마나 어긋나는지 미리 재둔 교정표**로 보정하는 편이 낫다.

방법은 이 리포트가 한 그대로다: 신호마다 대량의 잡음 지도에서 **명목 vs 경험 오경보율 곡선**(그림 a)을 재둔다. 그러면 '진짜 만분의 1을 얻으려면 문턱을 얼마로 줘야 하나'를 신호별로 역산할 수 있다. 이 교정표를 거친 뒤에야 세 신호가 **같은 진짜 오경보율** 위에 서고, 그제서야 탐지율 비교가 공정해진다.

### 그리고 — 깊은 오경보율일수록 더 조심

배율은 명목 오경보율이 깊어질수록(더 드문 오경보를 요구할수록) 더 커진다. 잡음의 꼬리가 CFAR 이 가정하는 것보다 두껍기 때문이다:

| 파형 | 배율 @ $10^{-4}$ | 배율 @ $10^{-6}$ |
|---|---|---|
| **WiFi 80MHz** | 1.45배 | **2.66배** |
| **LTE 20MHz** | 2.66배 | **5.67배** |
| **5G NR 100MHz** | 1.52배 | **2.13배** |

> 백만분의 1을 요구하면 LTE 는 실제로 **5.7배**까지 헛발화한다. 깊은 오경보율에서 운용할 계획이라면 교정은 더더욱 필수다.

---

### 이 리포트가 세운 것

- **검출기의 산수는 정확하다** — 이상적 잡음에서 눈금이 맞는다(§2.1).
- **실제 지도에서는 신호마다 다르게 어긋난다** — WiFi 1.45 · LTE 2.66 · 5G 1.52배(§2.2).
- **원인은 이웃 칸 상관** — 도플러축 Hann(공통) × 거리축 과표본(대역이 좁을수록). 끄면 복구된다(§2.3·§2.4).
- **검출단은 pyAPRiL 표준 CA-CFAR, 처방은 신호별 교정표** — 그래야 공정한 벤치마크가 가능하다(§4).

> **다음 리포트: report11 — 검출기 교정 ②: 저속 표적·거리 분해능.** 오경보율은 검출기가 지켜야 할 여러 약속 중 하나일 뿐이다. 신호 대역폭과 송수신 기하가 정하는 원리적 문턱 — 저속 표적 맹점·거리 분해능·위치 관측성 — 위에서 드론이 실제로 잡히는지, 남은 눈금들을 마저 맞춘다.

> _(멀리 보는 한 줄: 이 교정들이 끝나면, 탐지(거리+속도)를 넘어 **추후 트래킹까지 확장**할 토대가 된다.)_